In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_curve
from itertools import combinations
import pandas as pd
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [25]:
DEVICE      = "cpu"
NUM_CLASSES = 40
IMG_HEIGHT  = 112
IMG_WIDTH   = 92

ORL_FULL_PATH    = "dataset/Training"
CLAHE_FULL_X2    = "dataset/CLAHE/Training_CLAHE_escala2"
CLAHE_FULL_X4    = "dataset/CLAHE/Training_CLAHE_escala4"
SR_FULL_X2       = "dataset/Super_resolution/Training_SR_escala2"
SR_FULL_X4       = "dataset/Super_resolution/Training_SR_escala4"

In [16]:
full_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

def cargar_dataset_completo(path):
    dataset = datasets.ImageFolder(path, transform=full_transform)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    imagenes, etiquetas = [], []
    for img, label in loader:
        imagenes.append(img.squeeze(0))
        etiquetas.append(label.item())
    print(f"  Cargadas: {len(imagenes)} imágenes | {len(dataset.classes)} sujetos")
    return imagenes, etiquetas

In [17]:
def extraer_embeddings_mobilefacenet(model, imagenes):

    model.eval()
    embeddings = []

    with torch.no_grad():

        for img in imagenes:

            x = img.unsqueeze(0).to(DEVICE)

            x = model.stem(x)
            x = model.bottlenecks(x)
            x = model.conv_512(x)
            x = model.gdconv(x)
            x = model.embedding(x)

            x = F.normalize(x, p=2, dim=1)

            embeddings.append(
                x.squeeze(0).cpu().numpy()
            )

    return np.array(embeddings)


def extraer_embeddings_mobilefacenet2(model, imagenes):

    model.eval()
    embeddings = []

    with torch.no_grad():

        for img in imagenes:

            x = img.unsqueeze(0).to(DEVICE)

            x = model.stem(x)
            x = model.bottlenecks(x)
            x = model.conv_512(x)
            x = model.gdconv(x)
            x = model.embedding(x)
            x = model.dropout(x)

            x = F.normalize(x, p=2, dim=1)

            embeddings.append(
                x.squeeze(0).cpu().numpy()
            )

    return np.array(embeddings)

# Arquitectura MobilFacenet 

In [18]:
class ConvBN(nn.Module):
    def __init__(self, in_c, out_c, kernel, stride=1, padding=1, activation=True):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel, stride=stride,
                      padding=padding, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU6(inplace=True) if activation else nn.Identity()
        )
    def forward(self, x):
        return self.block(x)


class DepthwiseBN(nn.Module):
    def __init__(self, channels, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, stride=stride,
                      padding=1, groups=channels, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU6(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class Bottleneck(nn.Module):
    def __init__(self, in_c, out_c, stride, expansion):
        super().__init__()
        expanded = in_c * expansion
        self.use_residual = (stride == 1 and in_c == out_c)

        layers_list = []
        if expansion != 1:
            layers_list.append(ConvBN(in_c, expanded, 1, padding=0))
        layers_list += [
            DepthwiseBN(expanded, stride=stride),
            nn.Conv2d(expanded, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c)
        ]
        self.block = nn.Sequential(*layers_list)

    def forward(self, x):
        out = self.block(x)
        if self.use_residual:
            out = out + x
        return out

In [19]:
class MobileFaceNet(nn.Module):
    """
    MobileFaceNet adaptado para ORL Dataset
    Input : (B, 1, 112, 92) — escala de grises
    Output: (B, num_classes)
    """
    def __init__(self, num_classes=NUM_CLASSES, embedding_dim=128):
        super().__init__()

        # ---- Stem ----
        self.stem = nn.Sequential(
            ConvBN(1, 64, 3, stride=2, padding=1),    # → (56, 46)
            DepthwiseBN(64, stride=1)
        )

        # ---- Bottleneck blocks ----
        # (expansion, out_channels, repeticiones, stride_primera)
        cfg = [
            (2,  64,  5, 2),   # → (28, 23)
            (4, 128,  1, 2),   # → (14, 12)
            (2, 128,  6, 1),   # → (14, 12)
            (4, 128,  1, 2),   # → ( 7,  6)
            (2, 128,  2, 1),   # → ( 7,  6)
        ]

        blocks = []
        in_c = 64
        for t, c, n, s in cfg:
            for i in range(n):
                stride = s if i == 0 else 1
                blocks.append(Bottleneck(in_c, c, stride, t))
                in_c = c
        self.bottlenecks = nn.Sequential(*blocks)

        # ---- Conv 1×1 ----
        self.conv_512 = ConvBN(128, 512, 1, padding=0)   # → (7, 6, 512)
        self.gdconv = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=(7, 6),
            groups=512, bias=False),       
            nn.GroupNorm(num_groups=32, num_channels=512) 
        )

        # ---- Embedding 1×1 ----
        self.embedding = nn.Sequential(
            nn.Conv2d(512, embedding_dim, 1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=embedding_dim),
            nn.Flatten()                                  # → 128-D
        )

        # ---- Clasificador ----
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.bottlenecks(x)
        x = self.conv_512(x)
        x = self.gdconv(x)
        x = self.embedding(x)
        return self.classifier(x)


model = MobileFaceNet().to(DEVICE)
print(model)

# Verificar shapes con un batch ficticio
dummy = torch.zeros(1, 1, IMG_HEIGHT, IMG_WIDTH).to(DEVICE)
out   = model(dummy)
print(f"\nShape de salida con input (1,1,112,92): {out.shape}")  # → (1, 40)

MobileFaceNet(
  (stem): Sequential(
    (0): ConvBN(
      (block): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
    )
    (1): DepthwiseBN(
      (block): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
    )
  )
  (bottlenecks): Sequential(
    (0): Bottleneck(
      (block): Sequential(
        (0): ConvBN(
          (block): Sequential(
            (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
     

# MobilFacenet2 ArcFace

In [20]:
class ConvBN_x2(nn.Module):
    def __init__(self, in_c, out_c, kernel, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel, stride=stride,
                      padding=padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_c),
            nn.PReLU(out_c)   # PReLU por canal en lugar de ReLU
        )
    def forward(self, x):
        return self.conv(x)

class DepthwiseBN_x2(nn.Module):
    def __init__(self, channels, stride=1):
        super().__init__()
        self.dw = nn.Sequential(
            nn.Conv2d(channels, channels, 3, stride=stride,
                      padding=1, groups=channels, bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(channels)
        )
    def forward(self, x):
        return self.dw(x)

class Bottleneck_x2(nn.Module):
    def __init__(self, in_c, out_c, stride, expansion):
        super().__init__()
        mid_c = in_c * expansion
        self.block = nn.Sequential(
            # Expand
            nn.Conv2d(in_c, mid_c, 1, bias=False),
            nn.BatchNorm2d(mid_c),
            nn.PReLU(mid_c),
            # Depthwise
            nn.Conv2d(mid_c, mid_c, 3, stride=stride,
                      padding=1, groups=mid_c, bias=False),
            nn.BatchNorm2d(mid_c),
            nn.PReLU(mid_c),
            # Project (sin activación al final)
            nn.Conv2d(mid_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.use_residual = (stride == 1 and in_c == out_c)

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        return self.block(x)

In [21]:
import torch.nn.functional as F

class ArcFaceLoss_x2(nn.Module):
    def __init__(self, in_features, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        emb_norm = F.normalize(embeddings, dim=1)
        w_norm   = F.normalize(self.weight, dim=1)

        cosine = F.linear(emb_norm, w_norm).clamp(-1 + 1e-7, 1 - 1e-7)
        theta  = torch.acos(cosine)

        target_logits = torch.cos(theta + self.m)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        output = self.s * (one_hot * target_logits + (1 - one_hot) * cosine)
        return F.cross_entropy(output, labels)

In [22]:
class MobileFaceNet2_x2(nn.Module):
    """
    MobileFaceNet con PReLU + ArcFace
    El classifier se elimina — ArcFaceLoss lo reemplaza
    """
    def __init__(self, embedding_dim=128):
        super().__init__()

        # Stem con stride=1 en primera conv (fiel al paper)
        self.stem = nn.Sequential(
            ConvBN_x2(1, 64, 3, stride=1, padding=1),   # stride=1, preserva resolución
            DepthwiseBN_x2(64, stride=1)
        )

        cfg = [
            (2,  64,  5, 2),
            (4, 128,  1, 2),
            (2, 128,  6, 1),
            (4, 128,  1, 2),
            (2, 128,  2, 1),
        ]

        blocks = []
        in_c = 64
        for t, c, n, s in cfg:
            for i in range(n):
                stride = s if i == 0 else 1
                blocks.append(Bottleneck_x2(in_c, c, stride, t))
                in_c = c
        self.bottlenecks = nn.Sequential(*blocks)

        self.conv_512 = ConvBN_x2(128, 512, 1, padding=0)

        self.gdconv = nn.Sequential(
        nn.Conv2d(512, 512, kernel_size=(14, 12), groups=512, bias=False),
        nn.GroupNorm(num_groups=32, num_channels=512)
        )

        self.embedding = nn.Sequential(
            nn.Conv2d(512, embedding_dim, 1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=embedding_dim),
            nn.Flatten()
        )

        self.dropout = nn.Dropout(p=0.4)
        # Sin self.classifier — ArcFace lo maneja

    def forward(self, x):
        x = self.stem(x)
        x = self.bottlenecks(x)
        x = self.conv_512(x)
        x = self.gdconv(x)
        x = self.embedding(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return x   # retorna embedding 128-D normalizado
model = MobileFaceNet2_x2().to(DEVICE)
print(model)

MobileFaceNet2_x2(
  (stem): Sequential(
    (0): ConvBN_x2(
      (conv): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): PReLU(num_parameters=64)
      )
    )
    (1): DepthwiseBN_x2(
      (dw): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): PReLU(num_parameters=64)
      )
    )
  )
  (bottlenecks): Sequential(
    (0): Bottleneck_x2(
      (block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): PReLU(num_parameters=128)
        (3): Conv2d(128, 128, kernel_size=(3, 3), stride

In [23]:
def generar_pares(embeddings, etiquetas, n_impostores=10000, seed=42):
    np.random.seed(seed)
    etiquetas = np.array(etiquetas)
    n = len(embeddings)
    similitudes, labels_pares = [], []

    # Pares genuinos
    for sujeto in np.unique(etiquetas):
        indices = np.where(etiquetas == sujeto)[0]
        for i, j in combinations(indices, 2):
            sim = float(np.dot(embeddings[i], embeddings[j]))
            similitudes.append(sim)
            labels_pares.append(1)

    print(f"  Pares genuinos   : {len(similitudes)}")

    # Pares impostores
    count, intentos = 0, 0
    while count < n_impostores and intentos < n_impostores * 10:
        i, j = np.random.randint(0, n, size=2)
        if etiquetas[i] != etiquetas[j]:
            sim = float(np.dot(embeddings[i], embeddings[j]))
            similitudes.append(sim)
            labels_pares.append(0)
            count += 1
        intentos += 1

    print(f"  Pares impostores : {count}")
    return np.array(similitudes), np.array(labels_pares)


def calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.01):
    fpr, tpr, _ = roc_curve(labels_pares, similitudes)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2)

    indices_far = np.where(fpr <= far_objetivo)[0]
    tar = float(tpr[indices_far[-1]]) if len(indices_far) > 0 else 0.0

    return eer * 100, tar * 100

# Visualización de métricas

In [26]:
experimentos = [
    # (nombre, checkpoint, dataset_path, arquitectura)
    # arquitectura: "ce" = MobileFaceNet, "arcface" = MobileFaceNet2
    ("ORL original — CE",       "best_mobilefacenet_orl.pth",                    ORL_FULL_PATH, "ce"),
    ("CLAHE x2 — CE",           "best_mobilefacenet_experimento2_clahe_x2.pth",  CLAHE_FULL_X2, "ce"),
    ("CLAHE x4 — CE",           "best_mobilefacenet_clahe_x4.pth",               CLAHE_FULL_X4, "ce"),
    ("SR x2 — CE",              "best_mobilefacenet_experimento2_sr_x2.pth",     SR_FULL_X2,    "ce"),
    ("SR x4 — CE",              "best_mobilefacenet_experimento2_sr_x4.pth",     SR_FULL_X4,    "ce"),
    ("CLAHE x2 — ArcFace",      "best_mobilefacenet_arcface_experimento3_clahe_x2.pth",       CLAHE_FULL_X2, "arcface"),
    ("CLAHE x4 — ArcFace",      "best_mobilefacenet_arcface_clahe_x4.pth",       CLAHE_FULL_X4, "arcface"),
    ("SR x2 — ArcFace",         "best_mobilefacenet_arcface_experimento3_sr_x2.pth",          SR_FULL_X2,    "arcface"),
    ("SR x4 — ArcFace",         "best_mobilefacenet_arcface_experimento3_sr_x4.pth",          SR_FULL_X4,    "arcface"),
]

resultados = []

for nombre, ckpt_path, dataset_path, arch in experimentos:
    print(f"\n{'='*60}")
    print(f"Evaluando : {nombre}")
    print(f"Checkpoint: {ckpt_path}")

    # Cargar modelo según arquitectura
    if arch == "ce":
        model = MobileFaceNet(num_classes=NUM_CLASSES).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        extractor = extraer_embeddings_mobilefacenet
    else:
        model = MobileFaceNet2_x2(embedding_dim=128).to(DEVICE)
        ckpt  = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])   # solo backbone, sin pesos ArcFace
        extractor = extraer_embeddings_mobilefacenet2

    # Cargar dataset y extraer embeddings
    imagenes, etiquetas = cargar_dataset_completo(dataset_path)
    embeddings = extractor(model, imagenes)
    print(f"  Embeddings: {embeddings.shape}")

    # Pares y métricas
    similitudes, labels_pares = generar_pares(embeddings, etiquetas, n_impostores=10000)
    eer, tar_1  = calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.01)
    _,   tar_01 = calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.001)

    print(f"  EER            : {eer:.2f}%")
    print(f"  TAR @ FAR=1%   : {tar_1:.2f}%")
    print(f"  TAR @ FAR=0.1% : {tar_01:.2f}%")

    resultados.append({
        "Experimento":        nombre,
        "EER (%)":            round(eer, 2),
        "TAR @ FAR=1% (%)":  round(tar_1, 2),
        "TAR @ FAR=0.1% (%)": round(tar_01, 2),
    })

df = pd.DataFrame(resultados)
print("\n" + "="*60)
print("TABLA RESUMEN — EER y TAR (MobileFaceNet / MobileFaceNet2)")
print("="*60)
print(df.to_string(index=False))


Evaluando : ORL original — CE
Checkpoint: best_mobilefacenet_orl.pth
  Cargadas: 360 imágenes | 40 sujetos
  Embeddings: (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
  EER            : 0.07%
  TAR @ FAR=1%   : 100.00%
  TAR @ FAR=0.1% : 99.93%

Evaluando : CLAHE x2 — CE
Checkpoint: best_mobilefacenet_experimento2_clahe_x2.pth
  Cargadas: 360 imágenes | 40 sujetos
  Embeddings: (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
  EER            : 2.16%
  TAR @ FAR=1%   : 95.21%
  TAR @ FAR=0.1% : 79.72%

Evaluando : CLAHE x4 — CE
Checkpoint: best_mobilefacenet_clahe_x4.pth
  Cargadas: 360 imágenes | 40 sujetos
  Embeddings: (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
  EER            : 0.48%
  TAR @ FAR=1%   : 99.72%
  TAR @ FAR=0.1% : 97.85%

Evaluando : SR x2 — CE
Checkpoint: best_mobilefacenet_experimento2_sr_x2.pth
  Cargadas: 360 imágenes | 40 sujetos
  Embeddings: (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
 

In [27]:
rank1_conocido = {
    "ORL original — CE":   100.00,
    "CLAHE x2 — CE":        97.50,
    "CLAHE x4 — CE":        55.00,
    "SR x2 — CE":           97.50,
    "SR x4 — CE":           82.50,
    "CLAHE x2 — ArcFace":   92.50,
    "CLAHE x4 — ArcFace":   85.00,
    "SR x2 — ArcFace":      97.50,
    "SR x4 — ArcFace":      95.00,
}

df["Rank-1 (%)"] = df["Experimento"].map(rank1_conocido)
df_final = df[["Experimento", "Rank-1 (%)", "EER (%)",
               "TAR @ FAR=1% (%)", "TAR @ FAR=0.1% (%)"]]

print("TABLA FINAL — MobileFaceNet (Rank-1 + EER + TAR)")
print("="*70)
print(df_final.to_string(index=False))

df_final.to_csv("resultados_eer_tar_mobilefacenet.csv", index=False)
print("\nGuardado en: resultados_eer_tar_mobilefacenet.csv")

TABLA FINAL — MobileFaceNet (Rank-1 + EER + TAR)
       Experimento  Rank-1 (%)  EER (%)  TAR @ FAR=1% (%)  TAR @ FAR=0.1% (%)
 ORL original — CE       100.0     0.07            100.00               99.93
     CLAHE x2 — CE        97.5     2.16             95.21               79.72
     CLAHE x4 — CE        55.0     0.48             99.72               97.85
        SR x2 — CE        97.5     0.00            100.00              100.00
        SR x4 — CE        82.5     0.00            100.00              100.00
CLAHE x2 — ArcFace        92.5     4.29             91.32               74.58
CLAHE x4 — ArcFace        85.0     3.47             94.10               75.97
   SR x2 — ArcFace        97.5     2.68             94.86               85.69
   SR x4 — ArcFace        95.0     0.87             99.31               95.42

Guardado en: resultados_eer_tar_mobilefacenet.csv
